In [ ]:
%matplotlib inline
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from adjustText import adjust_text
from scipy import stats as sp_stats

os.makedirs('figures', exist_ok=True)
df = pd.read_csv('all_data.csv')

def parse_size(s):
    if pd.isna(s): return np.nan
    s = str(s).strip()
    if s.endswith('B'): return float(s[:-1]) * 1e9
    elif s.endswith('M'): return float(s[:-1]) * 1e6
    elif s.endswith('K'): return float(s[:-1]) * 1e3
    return float(s)

df['size'] = df['model size'].apply(parse_size)
df['log_size'] = np.log10(df['size'])
df['log_time'] = np.log10(df['inference_time_s'].clip(lower=0.1))
df['log_co2_total'] = np.log10(df['CO2_total'].clip(lower=0.01))
df['marker_size'] = df['size'].apply(
    lambda x: 30 + np.clip((np.log10(x)-5)/5,0,1)*370 if not pd.isna(x) else 80)

# Baseline-relative metrics
bl = df[df['baseline?']==True][['task','performance','CO2_per_call']].rename(
    columns={'performance':'p0','CO2_per_call':'c0'})
df = df.merge(bl, on='task', how='left')
df['delta_perf'] = (df['performance'] - df['p0']) / df['p0']
df['co2_ratio'] = df['CO2_per_call'] / df['c0']
df.loc[df['baseline?']==True, 'delta_perf'] = 0.0
df['time_per_param'] = df['inference_time_s'] / df['size']

tasks = ['MatGen','MolGen','Retro','Forward','StructOpt','MDSim']
TC = {'MatGen':'tab:green','MolGen':'tab:cyan','Retro':'tab:blue',
      'Forward':'tab:orange','StructOpt':'tab:red','MDSim':'tab:purple'}
PM = {'MatGen':'mSUN','MolGen':'VUN (%)','Retro':'Top-50 (%)',
      'Forward':'Top-3 (%)','StructOpt':'CPS','MDSim':'MSD'}
AM = {'MLP':'o','GNN':'d','LM':'^','LLM':'*','VAE':'s','Diffusion':'X','Flow Matching':'P'}
leg_am = [Line2D([0],[0],marker=m,color='gray',linestyle='None',markersize=7,label=a) for a,m in AM.items()]

print(f"Loaded {len(df)} models across {len(tasks)} tasks")
df[['task','model','model type','performance','CO2_per_call','call_unit']].head(10)


# Section 1: Raw Data — Year-wise Trends

How are performance, model size, and carbon cost evolving over the years?

In [ ]:
# Fig 1a: Year vs Performance (per task)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for idx, task in enumerate(tasks):
    ax = axes.flat[idx]
    t = df[df['task']==task].sort_values('year')
    for _, r in t.iterrows():
        ax.scatter(r['year'], r['performance'], marker=AM.get(r['model type'],'o'),
                   s=r['marker_size'], color=TC[task], edgecolors='black', linewidth=0.5, zorder=3)
    texts = [ax.text(r['year'], r['performance'], r['model'], fontsize=6) for _, r in t.iterrows()]
    if texts: adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.4))
    ax.set_xlabel('Year'); ax.set_ylabel(PM[task]); ax.set_title(task, fontweight='bold')
    ax.grid(True, alpha=0.3)
legend = [Line2D([0],[0],marker=m,color='gray',linestyle='None',markersize=7,label=a) for a,m in AM.items()]
fig.legend(handles=legend, loc='lower center', ncol=7, fontsize=8, bbox_to_anchor=(0.5,-0.01))
fig.suptitle('Year vs Performance (marker size = model size)', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0.03,1,0.96])
plt.savefig('figures/1a_year_vs_performance.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Fig 1b: Year vs Model Size (per task)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for idx, task in enumerate(tasks):
    ax = axes.flat[idx]
    t = df[df['task']==task].sort_values('year')
    for _, r in t.iterrows():
        ax.scatter(r['year'], r['size'], marker=AM.get(r['model type'],'o'),
                   s=100, color=TC[task], edgecolors='black', linewidth=0.5, zorder=3)
    texts = [ax.text(r['year'], r['size'], r['model'], fontsize=6) for _, r in t.iterrows()]
    if texts: adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.4))
    ax.set_yscale('log'); ax.set_xlabel('Year'); ax.set_ylabel('Model Size (params)')
    ax.set_title(task, fontweight='bold'); ax.grid(True, alpha=0.3)
legend = [Line2D([0],[0],marker=m,color='gray',linestyle='None',markersize=7,label=a) for a,m in AM.items()]
fig.legend(handles=legend, loc='lower center', ncol=7, fontsize=8, bbox_to_anchor=(0.5,-0.01))
fig.suptitle('Year vs Model Size', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0.03,1,0.96])
plt.savefig('figures/1b_year_vs_model_size.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Fig 1c: Year vs CO2 per call (per task)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for idx, task in enumerate(tasks):
    ax = axes.flat[idx]
    t = df[df['task']==task].sort_values('year')
    unit = t['call_unit'].iloc[0]
    for _, r in t.iterrows():
        ax.scatter(r['year'], r['CO2_per_call'], marker=AM.get(r['model type'],'o'),
                   s=100, color=TC[task], edgecolors='black', linewidth=0.5, zorder=3)
    texts = [ax.text(r['year'], r['CO2_per_call'], r['model'], fontsize=6) for _, r in t.iterrows()]
    if texts: adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.4))
    ax.set_yscale('log'); ax.set_xlabel('Year'); ax.set_ylabel(f'CO₂/call ({unit})')
    ax.set_title(task, fontweight='bold'); ax.grid(True, alpha=0.3)
legend = [Line2D([0],[0],marker=m,color='gray',linestyle='None',markersize=7,label=a) for a,m in AM.items()]
fig.legend(handles=legend, loc='lower center', ncol=7, fontsize=8, bbox_to_anchor=(0.5,-0.01))
fig.suptitle('Year vs CO₂ per Call', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0.03,1,0.96])
plt.savefig('figures/1c_year_vs_co2_per_call.png', dpi=120, bbox_inches='tight')
plt.show()


# Section 2: Pareto Front Analysis

Relative performance gain vs relative CO₂ cost. Quadrants: **Dominant** (better + cheaper), **Tradeoff** (better + expensive), **Dominated** (worse + expensive), **Inverse** (worse + cheaper).

In [ ]:
# Pareto front helper
def get_pareto_front(points):
    """Find Pareto-optimal points: minimize x (CO2 ratio), maximize y (performance)."""
    if len(points) == 0:
        return np.array([])
    pts = np.vstack([[1, 0], points])  # include baseline at (1, 0)
    pareto_idx = []
    for i, (xi, yi) in enumerate(pts):
        dominated = False
        for j, (xj, yj) in enumerate(pts):
            if i != j and xj <= xi and yj >= yi and (xj < xi or yj > yi):
                dominated = True; break
        if not dominated:
            pareto_idx.append(i)
    pareto = pts[pareto_idx]
    return pareto[np.argsort(pareto[:, 0])]

# Fig 2a: Pareto quadrant — all tasks (2x3 grid)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for idx, task in enumerate(tasks):
    ax = axes.flat[idx]
    t_all = df[df['task']==task]
    t_nb = t_all[t_all['baseline?']==False].dropna(subset=['delta_perf','co2_ratio'])
    t_bl = t_all[t_all['baseline?']==True].iloc[0]

    ax.set_xscale('log')

    # Data-driven axis limits
    if len(t_nb) > 0:
        x_data = t_nb['co2_ratio']
        y_data = t_nb['delta_perf']
        xmin, xmax = x_data.min()*0.5, x_data.max()*2
        ymin, ymax = y_data.min()-0.1, y_data.max()+0.1
    else:
        xmin, xmax, ymin, ymax = 0.1, 10, -0.2, 0.2
    xmin, xmax = min(xmin, 0.5), max(xmax, 2)
    ymin, ymax = min(ymin, -0.1), max(ymax, 0.1)

    # Quadrant shading (boundary at x=1, y=0)
    ax.fill_between([xmin,1], 0, ymax, color='#c8e6c9', alpha=0.5, zorder=0)  # Dominant
    ax.fill_between([1,xmax], 0, ymax, color='#fff9c4', alpha=0.5, zorder=0)  # Tradeoff
    ax.fill_between([1,xmax], ymin, 0, color='#ffcdd2', alpha=0.5, zorder=0)  # Dominated
    ax.fill_between([xmin,1], ymin, 0, color='#e0e0e0', alpha=0.5, zorder=0)  # Inverse

    # Baseline at (1, 0)
    ax.scatter(1, 0, marker=AM.get(t_bl['model type'],'o'), s=160, color=TC[task],
               edgecolors='black', linewidth=1.5, zorder=6)

    # Non-baseline models
    for _, r in t_nb.iterrows():
        ax.scatter(r['co2_ratio'], r['delta_perf'], marker=AM.get(r['model type'],'o'),
                   s=r['marker_size'], color=TC[task], edgecolors='black', linewidth=0.7, zorder=5, alpha=0.9)

    # Pareto front line
    if len(t_nb) > 0:
        points = t_nb[['co2_ratio','delta_perf']].values
        pareto_pts = get_pareto_front(points)
        if len(pareto_pts) > 1:
            ax.step(pareto_pts[:,0], pareto_pts[:,1], where='post',
                    color='black', linestyle='--', linewidth=2, alpha=0.8, zorder=4)

    # Labels
    texts = [ax.text(1, 0, t_bl['model'], fontsize=7, fontweight='bold')]
    texts += [ax.text(r['co2_ratio'], r['delta_perf'], r['model'], fontsize=7) for _, r in t_nb.iterrows()]
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    ax.axhline(y=0, color='black', lw=1); ax.axvline(x=1, color='black', lw=1)
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
    ax.set_xlabel('CO₂ ratio (log scale)'); ax.set_ylabel('Δ Performance')
    ax.set_title(task, fontweight='bold', fontsize=12, color=TC[task])
    ax.grid(True, alpha=0.3)

arch_leg = [Line2D([0],[0],marker=m,color='gray',linestyle='None',markersize=7,label=a) for a,m in AM.items()]
quad_leg = [Patch(facecolor=c,alpha=0.5,edgecolor='black',label=q) for q,c in
            [('Dominant','#c8e6c9'),('Tradeoff','#fff9c4'),('Dominated','#ffcdd2'),('Inverse','#e0e0e0')]]
pareto_leg = [Line2D([0],[0],color='black',linestyle='--',linewidth=2,label='Pareto Front')]
fig.legend(handles=arch_leg+quad_leg+pareto_leg, loc='lower center', ncol=6, fontsize=8, bbox_to_anchor=(0.5,-0.01))
fig.suptitle('Pareto Front: Relative Performance vs CO₂ Ratio (marker size = model size)',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0.04,1,0.97])
plt.savefig('figures/2a_pareto_all_tasks.png', dpi=120, bbox_inches='tight')
plt.show()

# Summary
print("=== Quadrant Summary ===")
for task in tasks:
    t = df[(df['task']==task)&(df['baseline?']==False)].dropna(subset=['delta_perf','co2_ratio'])
    bl_name = df[(df['task']==task)&(df['baseline?']==True)].iloc[0]['model']
    dom = t[(t['co2_ratio']<=1)&(t['delta_perf']>0)]['model'].tolist()
    trd = t[(t['co2_ratio']>1)&(t['delta_perf']>0)]['model'].tolist()
    dtd = t[(t['co2_ratio']>1)&(t['delta_perf']<=0)]['model'].tolist()
    inv = t[(t['co2_ratio']<=1)&(t['delta_perf']<=0)]['model'].tolist()
    print(f"\n  {task} (baseline: {bl_name}): Dominant={dom}, Tradeoff={trd}, Dominated={dtd}, Inverse={inv}")


# Section 3: Carbon Emission Analysis

What drives CO₂? Decomposing the relationship between carbon emissions, model architecture, model size, and inference time.

In [ ]:
# Fig 3a: CO2 decomposition — log(time) vs log(CO2_total) with regression
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

v = df.dropna(subset=['log_time','log_co2_total','log_size'])

# Panel 1: time vs CO2 (strong predictor)
ax = axes[0]
for _, r in v.iterrows():
    ax.scatter(r['inference_time_s'], r['CO2_total'], marker=AM.get(r['model type'],'o'),
               s=80, color=TC[r['task']], edgecolors='black', linewidth=0.4, zorder=3, alpha=0.8)
# Regression line
slope, intercept = np.polyfit(v['log_time'].values, v['log_co2_total'].values, 1)
x_fit = np.linspace(v['log_time'].min()-0.3, v['log_time'].max()+0.3, 50)
ax.plot(10**x_fit, 10**(slope*x_fit+intercept), 'r--', lw=2, alpha=0.7)
r_val = np.corrcoef(v['log_time'], v['log_co2_total'])[0,1]
ax.text(0.05, 0.92, f'r = {r_val:.3f}\nR² = {r_val**2:.3f}', transform=ax.transAxes,
        fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Inference Time (s)', fontsize=11); ax.set_ylabel('CO₂ total (g)', fontsize=11)
ax.set_title('Inference Time → CO₂ (strong)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Panel 2: size vs CO2 (weak predictor)
ax = axes[1]
for _, r in v.iterrows():
    ax.scatter(r['size'], r['CO2_total'], marker=AM.get(r['model type'],'o'),
               s=80, color=TC[r['task']], edgecolors='black', linewidth=0.4, zorder=3, alpha=0.8)
slope2, intercept2 = np.polyfit(v['log_size'].values, v['log_co2_total'].values, 1)
x_fit2 = np.linspace(v['log_size'].min()-0.3, v['log_size'].max()+0.3, 50)
ax.plot(10**x_fit2, 10**(slope2*x_fit2+intercept2), 'r--', lw=2, alpha=0.7)
r_val2 = np.corrcoef(v['log_size'], v['log_co2_total'])[0,1]
ax.text(0.05, 0.92, f'r = {r_val2:.3f}\nR² = {r_val2**2:.3f}', transform=ax.transAxes,
        fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Model Size (params)', fontsize=11); ax.set_ylabel('CO₂ total (g)', fontsize=11)
ax.set_title('Model Size → CO₂ (weak)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Shared legend
task_leg = [Line2D([0],[0],marker='o',color=c,linestyle='None',markersize=7,label=t) for t,c in TC.items()]
arch_leg = [Line2D([0],[0],marker=m,color='gray',linestyle='None',markersize=7,label=a) for a,m in AM.items()]
fig.legend(handles=task_leg+arch_leg, loc='lower center', ncol=7, fontsize=7, bbox_to_anchor=(0.5,-0.02))
fig.suptitle('What Drives CO₂? Inference Time (R²=0.97) >> Model Size (R²=0.19)', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0.05,1,0.95])
plt.savefig('figures/3a_co2_decomposition.png', dpi=120, bbox_inches='tight')
plt.show()

# Multiple regression
from numpy.linalg import lstsq
X = np.column_stack([v['log_size'].values, v['log_time'].values, np.ones(len(v))])
y = v['log_co2_total'].values
coefs, _, _, _ = lstsq(X, y, rcond=None)
y_pred = X @ coefs
r2 = 1 - np.sum((y-y_pred)**2)/np.sum((y-y.mean())**2)
print(f"Multiple regression: log(CO2) = {coefs[0]:.3f}*log(size) + {coefs[1]:.3f}*log(time) + {coefs[2]:.3f}")
print(f"R² = {r2:.3f}")
print(f"Inference time is {abs(coefs[1]/coefs[0]):.0f}x more important than model size")


In [ ]:
# Fig 3b: Model Size vs Inference Time (per task) — showing no correlation
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for idx, task in enumerate(tasks):
    ax = axes.flat[idx]
    t = df[df['task']==task].dropna(subset=['size','inference_time_s'])
    for _, r in t.iterrows():
        ax.scatter(r['size'], r['inference_time_s'], marker=AM.get(r['model type'],'o'),
                   s=100, color=TC[task], edgecolors='black', linewidth=0.5, zorder=3)
    texts = [ax.text(r['size'], r['inference_time_s'], f"{r['model']}\n({r['model type']})", fontsize=5.5) for _, r in t.iterrows()]
    if texts: adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.4))
    # Regression + stats
    r_val, p_val = sp_stats.pearsonr(t['log_size'].values, t['log_time'].values)
    slope, intercept = np.polyfit(t['log_size'].values, t['log_time'].values, 1)
    xr = np.linspace(t['log_size'].min()-0.3, t['log_size'].max()+0.3, 50)
    ax.plot(10**xr, 10**(slope*xr+intercept), 'r--', lw=1.5, alpha=0.5)
    sig = '***' if p_val<0.001 else '**' if p_val<0.01 else '*' if p_val<0.05 else ' (n.s.)'
    ax.text(0.05, 0.92, f'r={r_val:+.2f}{sig}\np={p_val:.3f}', transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5), va='top')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Model Size (params)'); ax.set_ylabel('Inference Time (s)')
    ax.set_title(task, fontweight='bold'); ax.grid(True, alpha=0.3)
legend = [Line2D([0],[0],marker=m,color='gray',linestyle='None',markersize=7,label=a) for a,m in AM.items()]
fig.legend(handles=legend, loc='lower center', ncol=7, fontsize=8, bbox_to_anchor=(0.5,-0.01))
fig.suptitle('Model Size vs Inference Time — No Significant Correlation in Any Task', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0.03,1,0.96])
plt.savefig('figures/3b_size_vs_time.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Fig 3c: Computational density — time per parameter by architecture
df['time_per_param'] = df['inference_time_s'] / df['size']
arch_colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2']
arch_order = list(AM.keys())

fig, ax = plt.subplots(figsize=(10, 6))
data = [df[df['model type']==a]['time_per_param'].dropna().values for a in arch_order]
bp = ax.boxplot(data, tick_labels=arch_order, patch_artist=True, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='white', markersize=6))
for patch, color in zip(bp['boxes'], arch_colors):
    patch.set_facecolor(color); patch.set_alpha(0.6)
for i, arch in enumerate(arch_order):
    vals = df[df['model type']==arch]['time_per_param'].dropna().values
    ax.scatter(np.random.normal(i+1, 0.08, len(vals)), vals,
               alpha=0.6, s=40, color=arch_colors[i], edgecolors='black', linewidth=0.3, zorder=3)
ax.set_yscale('log')
ax.set_ylabel('Inference Time / Parameter (s/param)', fontsize=11)
ax.set_title('Computational Density by Architecture Type', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=20)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('figures/3c_computational_density.png', dpi=120, bbox_inches='tight')
plt.show()

# Print medians
print("=== Computational Density (median time/param) ===")
for a in arch_order:
    vals = df[df['model type']==a]['time_per_param'].dropna()
    if len(vals)>0:
        print(f"  {a:15s}  median={vals.median():.2e} s/param  (n={len(vals)})")
